# 17.4 - RAG Capstone

Status: VERIFIED

## What Are We Solving?

Build a Retrieval-Augmented Generation system. We chunk documents, create embeddings (using TF-IDF as a proxy), perform similarity search, and evaluate retrieval quality.

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)

documents = [
    'Machine learning is a subset of artificial intelligence that enables systems to learn from data.',
    'Deep learning uses neural networks with many layers to model complex patterns in data.',
    'Natural language processing allows computers to understand and generate human language.',
    'Computer vision enables machines to interpret and understand visual information from images.',
    'Reinforcement learning trains agents to make decisions by rewarding desired behaviors.',
    'Transfer learning reuses pre-trained models to solve new but related problems efficiently.',
    'Generative AI creates new content including text, images, code, and music.',
    'Data preprocessing cleans and transforms raw data into suitable formats for analysis.',
    'Feature engineering creates informative variables from raw data to improve model performance.',
    'Model evaluation assesses how well a trained model generalizes to unseen data.',
    'Overfitting occurs when a model learns noise instead of the underlying pattern.',
    'Cross-validation provides robust estimates of model performance across data splits.',
]
print(f'Corpus: {len(documents)} documents')

Corpus: 12 documents


In [2]:
# Document Chunking
def chunk_documents(docs, max_chunk_len=200, overlap=50):
    chunks = []
    for doc in docs:
        words = doc.split()
        for i in range(0, len(words), max_chunk_len - overlap):
            chunk = ' '.join(words[i:i + max_chunk_len])
            chunks.append({'text': chunk, 'source_doc': docs.index(doc),
                           'chunk_idx': i // (max_chunk_len - overlap)})
    return chunks

chunks = chunk_documents(documents)
print(f'Total chunks: {len(chunks)}')
for c in chunks[:3]:
    print(f'  Doc {c["source_doc"]} Chunk {c["chunk_idx"]}: "{c["text"][:60]}..."')

Total chunks: 12
  Doc 0 Chunk 0: "Machine learning is a subset of artificial intelligence that..."
  Doc 1 Chunk 0: "Deep learning uses neural networks with many layers to model..."
  Doc 2 Chunk 0: "Natural language processing allows computers to understand a..."


In [3]:
# Embedding + Similarity Search (TF-IDF proxy)
corpus = [c['text'] for c in chunks]
vectorizer = TfidfVectorizer(max_features=1000)
embeddings = vectorizer.fit_transform(corpus)
print(f'Embedding matrix: {embeddings.shape}')

def retrieve(query, top_k=3):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, embeddings).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    return [(chunks[i], float(sims[i])) for i in top_idx]

query = 'How do neural networks learn from data?'
results = retrieve(query, top_k=3)
print(f'\nQuery: "{query}"')
for rank, (chunk, score) in enumerate(results, 1):
    print(f'  {rank}. [score={score:.4f}] Doc {chunk["source_doc"]}: "{chunk["text"][:70]}..."')

Embedding matrix: (12, 106)

Query: "How do neural networks learn from data?"
  1. [score=0.3105] Doc 1: "Deep learning uses neural networks with many layers to model complex p..."
  2. [score=0.2593] Doc 0: "Machine learning is a subset of artificial intelligence that enables s..."
  3. [score=0.2007] Doc 9: "Model evaluation assesses how well a trained model generalizes to unse..."


In [4]:
# Retrieval Evaluation
queries = [
    ('neural networks deep learning', {1, 2, 6}),
    ('preprocessing data cleaning', {7, 8}),
    ('model evaluation testing', {9, 10, 11, 12}),
    ('artificial intelligence agents', {0, 4}),
]

precisions, recalls, mrr = [], [], []
for q, relevant_doc_ids in queries:
    res = retrieve(q, top_k=5)
    retrieved_docs = set(r[0]['source_doc'] for r in res)
    hits = len(retrieved_docs & relevant_doc_ids)
    precisions.append(hits / len(res))
    recalls.append(hits / len(relevant_doc_ids))
    for rank, (ch, _) in enumerate(res, 1):
        if ch['source_doc'] in relevant_doc_ids:
            mrr.append(1.0 / rank)
            break
    else:
        mrr.append(0.0)

print(f'Mean Precision: {np.mean(precisions):.4f}')
print(f'Mean Recall:    {np.mean(recalls):.4f}')
print(f'MRR:            {np.mean(mrr):.4f}')
print(f'F1:             {2*np.mean(precisions)*np.mean(recalls)/(np.mean(precisions)+np.mean(recalls)):.4f}')
print('VERIFICATION PASSED: Phase 17.4 complete')

Mean Precision: 0.4000
Mean Recall:    0.7708
MRR:            1.0000
F1:             0.5267
VERIFICATION PASSED: Phase 17.4 complete
